# Plotting a `Dataset`

This gallery walks through every way to plot a single raster with [`Dataset.plot`](../../reference/dataset/analysis.md) and `Dataset.animate`. pyramids owns no drawing code — each call builds a [cleopatra](https://github.com/serapeum-org/cleopatra) glyph — so the parameters below are the full matplotlib rendering surface: no-data handling, the renderer `kind`, colour scaling and colour bars, contours, relief shading, basemaps, per-cell labels, point overlays, multi-band selection, animation, and the new `backend=` seam.

Every section explains the idea, runs one short call on real data, and shows the figure.

## Setup

Inline figures, the imports, and the notebook-relative path to the bundled example data.

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np

from pyramids.dataset import Dataset
from pyramids.plot import (
    CellValues,
    ColorBar,
    ColorScaling,
    Contour,
    DataStyle,
    PointOverlay,
)

DATA = Path('../../../examples/data')

Load three real rasters: a filled **digital elevation model** of the Rhine basin (the main subject), a tiny flow-accumulation grid whose few cells are big enough to label, and a four-band precipitation stack for the multi-band and animation examples.

In [ ]:
dem = Dataset.read_file(str(DATA / 'dem' / 'DEM5km_Rhine_burned_fill.tif'))
small = Dataset.read_file(str(DATA / 'acc4000.tif'))
multiband = Dataset.read_file(str(DATA / 'geotiff' / 'noah-precipitation-1979.tif'))
{
    'dem': dem.no_data_value,
    'small': small.no_data_value,
    'multiband': multiband.no_data_value,
}

## No-data handling

Each raster carries a **no-data value** (printed above) marking cells with no measurement — the basin's surrounding rectangle here. `plot` reads it from the file and masks those cells automatically, so they render transparent (the white border), never as a real value that would wreck the colour range. The quickstart below shows that masking in action; the colour bar spans only the real elevations, not the no-data sentinel.

## Quickstart

The shortest call: `plot()` with no arguments masks no-data, picks sensible defaults, and returns a cleopatra `ArrayGlyph` wrapping the figure.

In [ ]:
dem.plot()

## Masking extra values: `exclude_value=`

Beyond the file's no-data, `exclude_value` masks one more value before plotting — here the flat zero cells of the accumulation grid, leaving only the drainage network.

In [ ]:
small.plot(exclude_value=0)

## The renderer: `kind=`

`kind` chooses how the array is drawn. `"auto"` (the default) picks per data; the explicit options are `imshow`, `pcolormesh`, `contour` and `contourf`.

In [ ]:
dem.plot(kind='pcolormesh')

In [ ]:
dem.plot(kind='contourf')

## Colour scaling: `color=`

`color` takes a typed `ColorScaling` spec controlling how values map to colours. A power norm stretches the low end; a boundary norm buckets values into discrete classes.

In [ ]:
dem.plot(color=ColorScaling.power(gamma=0.6))

In [ ]:
dem.plot(color=ColorScaling.boundary(bounds=[0, 200, 500, 1000, 2000, 3300]))

## Colour bar: `colorbar=`

Pass `True`/`False` to toggle the bar, or a `ColorBar` spec to style and label it (orientation, label text, size, …).

In [ ]:
dem.plot(colorbar=ColorBar(label='elevation [m]', orientation='horizontal'))

## Contours: `contour=`

With a contour `kind`, a `Contour` spec sets the levels and optional line labels.

In [ ]:
dem.plot(kind='contourf', contour=Contour(levels=8, labels=True))

## Relief shading: `data_style=`

`DataStyle` applies cleopatra's styling — here a hillshade that adds relief from the elevations themselves.

In [ ]:
dem.plot(data_style=DataStyle(hillshade=True))

## Basemap: `basemap=`

`basemap=True` draws a web-tile backdrop under the raster (a provider name string picks a specific tile source). pyramids reprojects the tiles to the raster's CRS.

In [ ]:
dem.plot(basemap=True)

## Per-cell values: `cells=`

`CellValues` prints each cell's value on the map — only legible for a small grid, so this uses the tiny accumulation raster (its no-data cells stay blank).

In [ ]:
small.plot(cells=CellValues(show=True, size=7))

## Point overlay: `points=`

`PointOverlay` draws labelled points from a `(value, row, col)` array on top of the raster.

In [ ]:
pts = np.array([[50.0, 3, 4], [80.0, 6, 8]])
small.plot(points=PointOverlay(pts, color='black', size=60))

## Multi-band rasters: `band=`

The precipitation file holds four bands (daily global rainfall). Select which one to draw with `band=`; each is colour-mapped on its own, with no-data (ocean) masked. Rainfall is heavily skewed — most cells are near zero with rare high totals — so a plain linear scale with a diverging colormap washes everything to one colour. A sequential `cmap` plus a `power` `ColorScaling` reveals the real pattern. (The grouped `rgb_options=` argument instead composites three bands into one true-colour image — it is meant for genuine RGB imagery, so it is not shown on this precipitation stack.)

In [ ]:
multiband.plot(
    band=0,
    cmap='viridis',
    color=ColorScaling.power(gamma=0.4),
    colorbar=ColorBar(label='rainfall band 0 [mm]'),
)

In [ ]:
multiband.plot(
    band=3,
    cmap='viridis',
    color=ColorScaling.power(gamma=0.4),
    colorbar=ColorBar(label='rainfall band 3 [mm]'),
)

## Animation: `Dataset.animate(...)`

`animate` renders the chosen bands as successive frames and returns a glyph whose `anim` attribute is a matplotlib `FuncAnimation`. The same skew applies across the stack, so the sequential colormap and `power` norm carry over — without them every frame flattens to a single colour. Rendering the animation to inline JavaScript gives a playable clip in which the rainfall bands visibly change.

In [ ]:
from IPython.display import HTML

glyph = multiband.animate(
    bands=[0, 1, 2, 3],
    interval=500,
    cmap='viridis',
    color=ColorScaling.power(gamma=0.4),
)
HTML(glyph.anim.to_jshtml())

## Choosing a renderer with `backend=`

Every `plot` and `animate` method takes a keyword-only `backend=`. The default, `"matplotlib"`, draws through
[cleopatra](https://github.com/serapeum-org/cleopatra) — that is every figure above. Any other name is resolved at
call time through the `pyramids.plot_backends` entry-point group, with an optional fall-back to
[digitalearth](https://github.com/serapeum-org/Digital-Earth)'s richer tiers. pyramids declares **no dependency** on
any non-matplotlib renderer, so a bare install always plots through matplotlib.

| `backend=` | Renderer | Install |
|---|---|---|
| `"matplotlib"` (default) | cleopatra / matplotlib — static figures | included with the `[viz]` extra |
| `"interactive"` | digitalearth (HoloViz / Bokeh) | `pip install 'digitalearth[interactive]'` |
| `"3d"` | digitalearth (PyVista) | `pip install 'digitalearth[3d]'` |
| `"web"` | digitalearth (MapLibre / deck.gl) | `pip install 'digitalearth[web]'` |

Stating the default explicitly produces exactly the quickstart figure; the digitalearth tiers follow:

In [ ]:
dem.plot(backend='matplotlib')

### Interactive map

`backend="interactive"` returns a digitalearth `InteractiveMap` (HoloViz / Bokeh). It displays inline — pan and zoom it.

In [ ]:
# digitalearth provides the non-matplotlib tiers. Load the HoloViz (Bokeh) extension so the
# interactive map embeds inline, and a small helper that embeds a standalone HTML document (the web map and the
# 3-D scene export full HTML pages) inline via a data-URI iframe.
import base64
import os
import tempfile

import holoviews as hv
from IPython.display import HTML

hv.extension("bokeh")


def embed(html, height=520):
    b64 = base64.b64encode(html.encode("utf-8")).decode("ascii")
    return HTML(
        f'<iframe src="data:text/html;base64,{b64}" width="100%" '
        f'height="{height}" style="border:none"></iframe>'
    )

In [ ]:
dem.plot(backend="interactive")

### Web map

`backend="web"` returns a `WebMap` (MapLibre GL). Its `to_html()` is a full standalone page, embedded here in an iframe.

In [ ]:
embed(dem.plot(backend="web").to_html(), height=520)

### 3-D scene

`backend="3d"` returns a `Scene3D` (PyVista). `export_html` writes a self-contained VTK.js page, embedded below — drag to orbit the terrain.

In [ ]:
scene = dem.plot(backend="3d")
scene_path = os.path.join(tempfile.gettempdir(), "de_scene3d.html")
scene.export_html(scene_path)
embed(open(scene_path, encoding="utf-8").read(), height=560)

Each call returned **digitalearth's own object** (`InteractiveMap` / `WebMap` /
`Scene3D`), not a matplotlib figure — the seam resolved the backend through the `pyramids.plot_backends` entry
point and handed the dataset to digitalearth. The matplotlib-only styling (the typed `color` / `contour` /
`cells` / `data_style` groups, the `points` overlay, a `ColorBar` instance, and read/render directives such as
`overview`) is **rejected** with a `ValueError` on a non-matplotlib backend, so a style is never silently
dropped — the calls above are deliberately bare so every backend can honour them.

## Takeaway

One `plot` call covers the whole static-raster surface — no-data masking, renderer, colour scaling, bars, contours, relief, basemaps, cell labels, point overlays and band selection — and `animate` turns a band stack into a movie. `backend=` keeps every one of these on matplotlib by default while routing to any registered renderer (digitalearth's interactive, 3-D and web tiers) on request.